In [ ]:
import numpy as np
import pandas as pd
import re
from sklearn import preprocessing
from sklearn.impute import KNNImputer
import math
import import_ipynb
from cuzick_test import cuzick_test
from scipy.stats import ttest_ind,f_oneway
import matplotlib.pyplot as plt
import scipy.io as sio

In [ ]:
def sig_star(p):
    star = ""
    if p <= 0.01:
        star = "***"
    elif p <= 0.05:
        star = "*"
    return star 

In [ ]:
def isNaN(num):
    return num != num

In [ ]:
criteria_df = pd.read_excel('..\medical_data\study_criteria_table_label_V6.xlsx')

In [ ]:
luna_df = pd.read_csv('../eeg_data/combined_luna_output.csv')

In [ ]:
eeg_df = pd.read_csv(r'../eeg_data/eeg_list.csv')
ID_df = eeg_df[['FolderName','SignalName']]
ID_df = ID_df.dropna()
ID_df['ID'] = [row['SignalName'].lower()[:-4] for index,row in ID_df.iterrows()]

In [ ]:
ID_df

In [ ]:
luna_df['L'] = [ re.findall('[OCF]',row['CH'])[0] for index,row in luna_df.iterrows()]

In [ ]:
luna_df = luna_df[['ID', 'L', 'F', 'AMP', 'CHIRP', 'COUPL_ANGLE', 'COUPL_MAG',
       'COUPL_OVERLAP', 'COUPL_PV', 'DENS', 'DISPERSION', 'DISPERSION_P',
       'DUR', 'FFT', 'FRQ', 'FWHM', 'ISA_M', 'ISA_S', 'ISA_T', 'MINS', 'N',
       'N01', 'N02', 'NE', 'NOSC', 'Q', 'SYMM', 'SYMM2', 'SO', 'SO_AMP',
       'SO_DUR', 'SO_NEG_DUR', 'SO_P2P', 'SO_POS_DUR', 'SO_RATE',
       'SO_SLOPE_NEG1', 'SO_SLOPE_NEG2', 'SO_SLOPE_POS1', 'SO_SLOPE_POS2',
       'SO_TH_NEG', 'SO_TH_P2P']]

In [ ]:
average_luna_df = luna_df.drop_duplicates(subset=['ID','L'])

In [ ]:
average_luna_df 

In [ ]:
average_luna_df =  average_luna_df.reset_index(drop=True)

In [ ]:
c = ['F', 'AMP', 'CHIRP', 'COUPL_ANGLE', 'COUPL_MAG',
       'COUPL_OVERLAP', 'COUPL_PV', 'DENS', 'DISPERSION', 'DISPERSION_P',
       'DUR', 'FFT', 'FRQ', 'FWHM', 'ISA_M', 'ISA_S', 'ISA_T', 'MINS', 'N',
       'N01', 'N02', 'NE', 'NOSC', 'Q', 'SYMM', 'SYMM2', 'SO', 'SO_AMP',
       'SO_DUR', 'SO_NEG_DUR', 'SO_P2P', 'SO_POS_DUR', 'SO_RATE',
       'SO_SLOPE_NEG1', 'SO_SLOPE_NEG2', 'SO_SLOPE_POS1', 'SO_SLOPE_POS2',
       'SO_TH_NEG', 'SO_TH_P2P']

In [ ]:
#Average channels in same lobe
for index,row in average_luna_df.iterrows():
    L = row['L']
    ID = row['ID']
    df = luna_df[(luna_df['ID'] == ID)&(luna_df['L'] == L)][c]
    means = df.mean()
    average_luna_df.iloc[index][:] = [ID]+ [L]+ list(means)

In [ ]:
average_luna_df

In [ ]:
ID_df['ID'] = [row['ID'].replace(' ','_') for index,row in ID_df.iterrows()]

In [ ]:
average_luna_df = average_luna_df.merge(ID_df,on=['ID'],how='left')

In [ ]:
average_luna_df = average_luna_df[average_luna_df.columns[1:]]

In [ ]:
average_luna_df = average_luna_df [['FolderName','L', 'F', 'AMP', 'CHIRP', 'COUPL_ANGLE', 'COUPL_MAG', 'COUPL_OVERLAP',
       'COUPL_PV', 'DENS', 'DISPERSION', 'DISPERSION_P', 'DUR', 'FFT', 'FRQ',
       'FWHM', 'ISA_M', 'ISA_S', 'ISA_T', 'MINS', 'N', 'N01', 'N02', 'NE',
       'NOSC', 'Q', 'SYMM', 'SYMM2', 'SO', 'SO_AMP', 'SO_DUR', 'SO_NEG_DUR',
       'SO_P2P', 'SO_POS_DUR', 'SO_RATE', 'SO_SLOPE_NEG1', 'SO_SLOPE_NEG2',
       'SO_SLOPE_POS1', 'SO_SLOPE_POS2', 'SO_TH_NEG', 'SO_TH_P2P' ]]

In [ ]:
average_luna_df.to_csv('average_luna_df.csv',index=False)

In [ ]:
average_luna_df = pd.read_csv('average_luna_df.csv')

In [ ]:
average_luna_df

# Create Spindle Features Table

In [ ]:
matched_df = pd.read_csv('matched_df.csv')

In [ ]:
matched_average_luna_df = matched_df[['FolderName']].merge(average_luna_df,on=['FolderName']).drop_duplicates(subset=['FolderName','L'])

In [ ]:
matched_average_luna_df = matched_average_luna_df.reset_index(drop=True)

In [ ]:
spindle_df = pd.DataFrame(columns=['Feature','L','DM_mean','DM_std','MCI_mean','MCI_std','CN_mean','CN_std','Cuzick_p','ANOVA_p'])
for L in ['F','C','O']:
    L_df =  pd.DataFrame(columns=['Feature','L','DM_mean','DM_std','MCI_mean','MCI_std','CN_mean','CN_std','Cuzick_p','ANOVA_p'])
    df = matched_average_luna_df[matched_average_luna_df['L'] == L]
    i = 0
    for f in c: 
        L_df.at[i,'Feature'] = f
        L_df.at[i,'L'] = L
        DM_f = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'Dementia']['FolderName'])][f].dropna()
        MCI_f = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'MCI']['FolderName'])][f].dropna()
        CN_f = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'No Dementia']['FolderName'])][f].dropna()
        
        L_df.at[i,'DM_mean'] = DM_f.mean()
        L_df.at[i,'DM_std'] = DM_f.std()
        L_df.at[i,'MCI_mean'] = MCI_f.mean()
        L_df.at[i,'MCI_std'] = MCI_f.std()
        L_df.at[i,'CN_mean'] = CN_f.mean()
        L_df.at[i,'CN_std'] = CN_f.std()

        stat,p = f_oneway(DM_f,MCI_f,CN_f)
        L_df.at[i,'ANOVA_p'] = p
        t,p = cuzick_test([DM_f.to_list(),MCI_f.tolist(),CN_f.tolist()])
        
        L_df.at[i,'Cuzick_p'] = p
        i += 1
    spindle_df = pd.concat([spindle_df,L_df])

In [ ]:
spindle_df['sig_trend'] = ['Yes' if row['Cuzick_p'] <0.05 else 'No' for index,row in spindle_df.iterrows()]

In [ ]:
spindle_df.sort_values(by=['L','Cuzick_p']).to_csv('spindle_features_df.csv',index=False)

In [ ]:
spindle_df.sort_values(by=['L','Cuzick_p'])

# Dementia Group

In [ ]:
criteria_df[criteria_df['Predicted_Stage'] == 'Dementia']['Age'].mean()

In [ ]:
len(criteria_df[(criteria_df['Predicted_Stage'] == 'Dementia') & (criteria_df['Sex'] == 'Male')])

In [ ]:
len(criteria_df[(criteria_df['Predicted_Stage'] == 'Dementia') & (criteria_df['Sex'] == 'Female')])

In [ ]:
dementia_df = average_luna_df[average_luna_df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'Dementia']['FolderName'])]

In [ ]:
dementia_df = dementia_df.reset_index(drop=True)
dementia_df

In [ ]:
#Not Normalized Means
mean_features = dementia_df[features].mean()
mean_features

In [ ]:
#normalization
X = dementia_df[features].values
X = KNNImputer(n_neighbors=10).fit_transform(X)
X = preprocessing.normalize(X)
normalized_dementia_df = dementia_df.copy()
normalized_dementia_df.iloc[:,3:] =X

In [ ]:
mean_features =normalized_dementia_df[features].mean()
mean_features

In [ ]:
#euclidian distance 

In [ ]:
#np.sum((features-mean_features)**2,axis=1)
#features.shape = (N,#features)
#mean_feature.shape(#feature,)

In [ ]:
# ED  = Euclidean distance 
for index,row in normalized_dementia_df.iterrows():
    ED = np.sum(np.array(row[features]-mean_features)**2)
    normalized_dementia_df.at[index,'ED'] = math.sqrt(ED)

In [ ]:
#Patient with smallest ED that is +- age 70 and male
#normalized_dementia_df.sort_values(by=['ED']).iloc[0]
normalized_dementia_df = normalized_dementia_df.sort_values(by=['ED'])

In [ ]:
Dementia_Example_FolderName = normalized_dementia_df[normalized_dementia_df['FolderName'].isin(criteria_df[(abs(criteria_df['Age']-70) <=2) & (criteria_df['Sex'] == 'Male')]['FolderName'])].iloc[0]['FolderName']

normalized_dementia_df[normalized_dementia_df['FolderName'].isin(criteria_df[(abs(criteria_df['Age']-70) <=2) & (criteria_df['Sex'] == 'Male')]['FolderName'])].iloc[0]

In [ ]:
dementia_df[(dementia_df['FolderName'] == 'McDonald_Robert_050316_2222.000') & (dementia_df['L'] == 'C' )].iloc[0]

In [ ]:
criteria_df[criteria_df['FolderName'] == 'McDonald_Robert_050316_2222.000'][['Age','Sex']]

# MCI

In [ ]:
MCI_df = average_luna_df[average_luna_df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'MCI']['FolderName'])]

In [ ]:
MCI_df = MCI_df.reset_index(drop=True)
MCI_df

In [ ]:
#normalization
#X = MCI_df[features].values
#Xstd = np.nanstd(X, axis=0)
#X = X/Xstd
#MCI_df.iloc[:,3:] =X

#normalization
X = MCI_df[features].values
X = KNNImputer(n_neighbors=10).fit_transform(X)
X = preprocessing.normalize(X)
normalized_MCI_df = MCI_df.copy()
normalized_MCI_df.iloc[:,3:] =X

In [ ]:
mean_features = MCI_df[features].mean()
mean_features

In [ ]:
#np.sum((features-mean_features)**2,axis=1)
#features.shape = (N,#features)
#mean_feature.shape(#feature,)

In [ ]:
# ED  = Euclidean distance 
for index,row in normalized_MCI_df.iterrows():
    ED = np.sum(np.array(row[features]-mean_features)**2)
    normalized_MCI_df.at[index,'ED'] = math.sqrt(ED)

In [ ]:
#Patient with smallest ED
#normalized_MCI_df.sort_values(by=['ED']).iloc[0]
MCI_Example_FolderName = normalized_MCI_df[normalized_MCI_df['FolderName'].isin(criteria_df[(abs(criteria_df['Age']-70) <=2) & (criteria_df['Sex'] == 'Male')]['FolderName'])].iloc[0]['FolderName']
normalized_MCI_df[normalized_MCI_df['FolderName'].isin(criteria_df[(abs(criteria_df['Age']-70) <=2) & (criteria_df['Sex'] == 'Male')]['FolderName'])].iloc[0]

In [ ]:
MCI_df[(MCI_df['FolderName'] == 'Melby_John_061715_2150.000') & (MCI_df['L'] == 'C' )].iloc[0]

# Non-Dementia

In [ ]:
Non_df = average_luna_df[average_luna_df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'No Dementia']['FolderName'])]

In [ ]:
Non_df = Non_df.reset_index(drop=True)

In [ ]:
Non_df[features].mean()

In [ ]:
#normalization
#normalization
X = Non_df[features].values
X = KNNImputer(n_neighbors=10).fit_transform(X)
X = preprocessing.normalize(X)
normalized_Non_df = Non_df.copy()
normalized_Non_df.iloc[:,3:] =X

In [ ]:
mean_features = Non_df[features].mean()
mean_features

In [ ]:
#np.sum((features-mean_features)**2,axis=1)
#features.shape = (N,#features)
#mean_feature.shape(#feature,)

In [ ]:
# ED  = Euclidean distance 
for index,row in normalized_Non_df.iterrows():
    ED = np.sum(np.array(row[features]-mean_features)**2)
    normalized_Non_df.at[index,'ED'] = math.sqrt(ED)

In [ ]:
#Patient with smallest ED
#normalized_Non_df.sort_values(by=['ED']).iloc[0]
Non_Example_FolderName = normalized_Non_df[normalized_Non_df['FolderName'].isin(criteria_df[(abs(criteria_df['Age']-70) <=2) & (criteria_df['Sex'] == 'Male')]['FolderName'])].iloc[0]['FolderName']
normalized_Non_df[normalized_Non_df['FolderName'].isin(criteria_df[(abs(criteria_df['Age']-70) <=2) & (criteria_df['Sex'] == 'Male')]['FolderName'])].iloc[0]

In [ ]:
Non_df[(Non_df['FolderName'] == normalized_Non_df.sort_values(by=['ED']).iloc[0]['FolderName']) & (Non_df['L'] == normalized_Non_df.sort_values(by=['ED']).iloc[0]['L'] )].iloc[0]

### Results

In [ ]:
print(dementia_df[(dementia_df['FolderName'] == normalized_dementia_df.sort_values(by=['ED']).iloc[0]['FolderName']) & (dementia_df['L'] == normalized_dementia_df.sort_values(by=['ED']).iloc[0]['L'] )].iloc[0])
print(MCI_df[(MCI_df['FolderName'] == normalized_MCI_df.sort_values(by=['ED']).iloc[0]['FolderName']) & (MCI_df['L'] == normalized_MCI_df.sort_values(by=['ED']).iloc[0]['L'] )].iloc[0])
print(Non_df[(Non_df['FolderName'] == normalized_Non_df.sort_values(by=['ED']).iloc[0]['FolderName']) & (Non_df['L'] == normalized_Non_df.sort_values(by=['ED']).iloc[0]['L'] )].iloc[0])

In [ ]:
print('Dementia Example:', Dementia_Example_FolderName)
print('MCI Example:', MCI_Example_FolderName)
print('Non-Dementia Example:',Non_Example_FolderName)

In [ ]:
dementia_example_df = dementia_df[(dementia_df['FolderName'] == Dementia_Example_FolderName) & (dementia_df['L'] == 'C' )].iloc[0]
MCI_example_df = MCI_df[(MCI_df['FolderName'] == MCI_Example_FolderName ) & (MCI_df['L'] == 'C' )].iloc[0]
Non_example_df = Non_df[(Non_df['FolderName'] == Non_Example_FolderName ) & (Non_df['L'] == 'C' )].iloc[0]

In [ ]:
spindle_dict = {}
for f in features:
    print(f)
    plt.plot([1,2.5,4],[dementia_example_df[f],MCI_example_df[f],Non_example_df[f]])
    plt.xticks([1,2.5,4],["Dementia",'MCI','Non-Dementia'])
    plt.xlim([0,5])
    plt.show()


# Group Analysis 

In [ ]:
average_luna_df

In [ ]:
row

In [ ]:
results_df = pd.DataFrame(columns=['Feature','L','ANOVA','Cuzick'])

In [ ]:
for L in ['F','C','O']:
    df = pd.DataFrame()
    df['Feature'] =list(average_luna_df.columns[2:-2])
    df['L'] = L
    results_df = results_df.append(df)

In [ ]:
results_df

In [ ]:
average_luna_df

In [ ]:
results_df = results_df.reset_index(drop=True)

In [ ]:
results_df

In [ ]:
#results1_df = pd.DataFrame(columns=['Feature','Dementia','MCI','Healthy','ANOVA','Cuzick Test','Dementia vs MCI','Dementia vs Healthy','MCI vs Healthy'])
i = 0
L='F'
df = average_luna_df[average_luna_df['L'] == L]
print('Feature Comparison for Frontal Lobe')
for var in average_luna_df.columns[2:-2]: 
    print(var)
    dementia_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'Dementia']['FolderName'])][var].dropna()
    MCI_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'MCI']['FolderName'])][var].dropna()
    Non_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'No Dementia']['FolderName'])][var].dropna()
    
    index = results_df.index[(results_df['Feature'] == var) & (results_df['L'] == L) ][0]
    print(index)
    print("Dementia:",dementia_var.mean(),"±",dementia_var.std(),'N=',len(dementia_var))
    print("MCI:",MCI_var.mean(),"±",MCI_var.std(),'N=',len(MCI_var))
    print("Healthy:",Non_var.mean(),"±",Non_var.std(),'N=',len(Non_var))
    
    
    stat,p = f_oneway(dementia_var,MCI_var,Non_var)
    print("ANOVA:",p,sig_star(p))
    results_df.at[index,'ANOVA'] = p
    t,p = cuzick_test([dementia_var.to_list(),MCI_var.tolist(),Non_var.tolist()])
    print("Cuzick:",p,sig_star(p))
    results_df.at[index,'Cuzick'] = p
    
    if p <= 0.05: 
        stat,p = ttest_ind(dementia_var,MCI_var)
        print("Dementia vs MCI:",p,sig_star(p))
        stat,p = ttest_ind(dementia_var,Non_var)
        print("Dementia vs Healthy",p,sig_star(p))
        stat,p = ttest_ind(MCI_var,Non_var)
        print("MCI vs Healthy",p,sig_star(p))
        x = ['Dementia', 'MCI', 'Healthy']
        y = [dementia_var.mean(),MCI_var.mean(),Non_var.mean()]
        #plt.boxplot([dementia_var.to_list(),MCI_var.tolist(),Non_var.tolist()],labels=x)
        plt.bar(x,y,yerr=[dementia_var.sem(),MCI_var.sem(),Non_var.sem()])
        plt.show()
    
    print("\n")
    i += 1

In [ ]:
#results1_df = pd.DataFrame(columns=['Feature','Dementia','MCI','Healthy','ANOVA','Cuzick Test','Dementia vs MCI','Dementia vs Healthy','MCI vs Healthy'])
i = 0
L='O'
df = average_luna_df[average_luna_df['L'] == L]
print('Feature Comparison for Frontal Lobe')
for var in average_luna_df.columns[2:-2]: 
    print(var)
    dementia_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'Dementia']['FolderName'])][var].dropna()
    MCI_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'MCI']['FolderName'])][var].dropna()
    Non_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'No Dementia']['FolderName'])][var].dropna()
    
    index = results_df.index[(results_df['Feature'] == var) & (results_df['L'] == L) ][0]
    
    print("Dementia:",dementia_var.mean(),"±",dementia_var.std(),'N=',len(dementia_var))
    print("MCI:",MCI_var.mean(),"±",MCI_var.std(),'N=',len(MCI_var))
    print("Healthy:",Non_var.mean(),"±",Non_var.std(),'N=',len(Non_var))
    
    
    stat,p = f_oneway(dementia_var,MCI_var,Non_var)
    print("ANOVA:",p,sig_star(p))
    results_df.at[index,'ANOVA'] = p
    t,p = cuzick_test([dementia_var.to_list(),MCI_var.tolist(),Non_var.tolist()])
    print("Cuzick:",p,sig_star(p))
    results_df.at[index,'Cuzick'] = p
    
    if p <= 0.05: 
        stat,p = ttest_ind(dementia_var,MCI_var)
        print("Dementia vs MCI:",p,sig_star(p))
        stat,p = ttest_ind(dementia_var,Non_var)
        print("Dementia vs Healthy",p,sig_star(p))
        stat,p = ttest_ind(MCI_var,Non_var)
        print("MCI vs Healthy",p,sig_star(p))
        x = ['Dementia', 'MCI', 'Healthy']
        y = [dementia_var.mean(),MCI_var.mean(),Non_var.mean()]
        #plt.boxplot([dementia_var.to_list(),MCI_var.tolist(),Non_var.tolist()],labels=x)
        plt.bar(x,y,yerr=[dementia_var.sem(),MCI_var.sem(),Non_var.sem()])
        plt.show()
    
    print("\n")
    i += 1

In [ ]:
#results1_df = pd.DataFrame(columns=['Feature','Dementia','MCI','Healthy','ANOVA','Cuzick Test','Dementia vs MCI','Dementia vs Healthy','MCI vs Healthy'])
i = 0
L='C'
df = average_luna_df[average_luna_df['L'] == L]
print('Feature Comparison for Frontal Lobe')
for var in average_luna_df.columns[2:-2]: 
    print(var)
    dementia_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'Dementia']['FolderName'])][var].dropna()
    MCI_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'MCI']['FolderName'])][var].dropna()
    Non_var = df[df['FolderName'].isin(criteria_df[criteria_df['Predicted_Stage'] == 'No Dementia']['FolderName'])][var].dropna()
    
    index = results_df.index[(results_df['Feature'] == var) & (results_df['L'] == L) ][0]
    
    print("Dementia:",dementia_var.mean(),"±",dementia_var.std(),'N=',len(dementia_var))
    print("MCI:",MCI_var.mean(),"±",MCI_var.std(),'N=',len(MCI_var))
    print("Healthy:",Non_var.mean(),"±",Non_var.std(),'N=',len(Non_var))
    
    
    stat,p = f_oneway(dementia_var,MCI_var,Non_var)
    print("ANOVA:",p,sig_star(p))
    results_df.at[index,'ANOVA'] = p
    t,p = cuzick_test([dementia_var.to_list(),MCI_var.tolist(),Non_var.tolist()])
    print("Cuzick:",p,sig_star(p))
    results_df.at[index,'Cuzick'] = p
    
    if p <= 0.05: 
        stat,p = ttest_ind(dementia_var,MCI_var)
        print("Dementia vs MCI:",p,sig_star(p))
        stat,p = ttest_ind(dementia_var,Non_var)
        print("Dementia vs Healthy",p,sig_star(p))
        stat,p = ttest_ind(MCI_var,Non_var)
        print("MCI vs Healthy",p,sig_star(p))
        x = ['Dementia', 'MCI', 'Healthy']
        y = [dementia_var.mean(),MCI_var.mean(),Non_var.mean()]
        #plt.boxplot([dementia_var.to_list(),MCI_var.tolist(),Non_var.tolist()],labels=x)
        plt.bar(x,y,yerr=[dementia_var.sem(),MCI_var.sem(),Non_var.sem()])
        plt.show()
    
    print("\n")
    i += 1

In [ ]:
results_df = results_df.sort_values(by=['Cuzick'])

In [ ]:
print('Significant Features:')
results_df[results_df['Cuzick']<0.05]['Feature'].unique()[:50]

In [ ]:
print('Insignificant Features:')
results_df[results_df['Cuzick']>0.05]['Feature'].unique()[:50]